In [0]:
%run .././utils/spark_config

In [0]:
# ── Cell 2: Download Kaggle → copy directly to ADLS ───────────────
import os
import json
import tempfile
import shutil
from datetime import datetime

# ── Build date-partitioned path ───────────────────────────────────
now   = datetime.now()
year  = now.strftime("%Y")
month = now.strftime("%m")
day   = now.strftime("%d")

dataset_folder = "creditcardfraud"
blob_path      = f"{dataset_folder}/year={year}/month={month}/day={day}/creditcard.csv"
adls_path      = paths["staging"] + blob_path

print("=" * 60)
print(" DATA SIMULATOR — CELL 2: STAGING INGESTION")
print("=" * 60)
print(f"📅 Partition   : year={year}/month={month}/day={day}")
print(f"📂 Target path : {adls_path}")

# ── Find a writable temp directory automatically ──────────────────
# tempfile.mkdtemp() tries multiple locations and returns
# the first one that is actually writable on this cluster
temp_dir = tempfile.mkdtemp(prefix="kaggle_")
print(f"✅ Writable temp directory found: {temp_dir}")

# ── Check if today's partition already exists in ADLS ─────────────
partition_exists = False
try:
    partition_folder = (paths["staging"] +
        f"{dataset_folder}/year={year}/month={month}/day={day}/")
    files      = dbutils.fs.ls(partition_folder)
    data_files = [f for f in files if not f.name.startswith("_")
                                   and not f.name.startswith(".")]
    if len(data_files) > 0:
        partition_exists = True
        print(f"\n✅ Today's partition already exists — skipping download")
        print(f"   Files: {[f.name for f in data_files]}")
except Exception:
    partition_exists = False
    print(f"\n📥 Partition not found — starting download...")

if not partition_exists:

    # ── Write Kaggle config to writable temp dir ──────────────────
    kaggle_config_dir = os.path.join(temp_dir, "kaggle_config")
    os.makedirs(kaggle_config_dir, exist_ok=True)

    kaggle_creds = {
        "username": dbutils.secrets.get("portfolio-scope", "kaggle-username"),
        "key":      dbutils.secrets.get("portfolio-scope", "kaggle-key")
    }

    config_file = os.path.join(kaggle_config_dir, "kaggle.json")
    with open(config_file, "w") as f:
        json.dump(kaggle_creds, f)
    os.chmod(config_file, 0o600)

    # Point Kaggle to writable config and download dirs
    os.environ["KAGGLE_CONFIG_DIR"] = kaggle_config_dir
    print(f"✅ Kaggle config written to {config_file}")

    # ── Monkey-patch os.utime to handle filesystem restrictions ───
    # Kaggle calls os.utime() to set file timestamps after download
    # Some Databricks cluster filesystems block this — patch it out
    original_utime = os.utime
    def safe_utime(path, times=None, **kwargs):
        try:
            original_utime(path, times, **kwargs)
        except (PermissionError, OSError):
            pass  # Silently ignore utime failures — file is still valid
    os.utime = safe_utime
    print(f"✅ os.utime patched — timestamp errors will be ignored")

    try:
        # ── Step 1: Download from Kaggle ──────────────────────────
        print("📥 Step 1: Downloading from Kaggle...")
        import kaggle
        kaggle.api.authenticate()
        kaggle.api.dataset_download_files(
            "mlg-ulb/creditcardfraud",
            path=temp_dir,
            unzip=True,
            quiet=False
        )

        # Find the downloaded CSV — could be named differently
        csv_file = None
        for f in os.listdir(temp_dir):
            if f.endswith(".csv"):
                csv_file = os.path.join(temp_dir, f)
                break

        if csv_file is None:
            all_files = os.listdir(temp_dir)
            raise FileNotFoundError(
                f"No CSV found in {temp_dir}. "
                f"Files present: {all_files}"
            )

        size_mb = round(os.path.getsize(csv_file) / (1024*1024), 2)
        print(f"✅ Download complete — {csv_file} ({size_mb} MB)")

        # ── Step 2: Copy raw file to ADLS ─────────────────────────
        print(f"📋 Step 2: Copying raw file to ADLS staging...")
        local_file_uri = f"file://{csv_file}"
        dbutils.fs.cp(local_file_uri, adls_path, recurse=False)

        print(f"✅ Raw file copied to ADLS")
        print(f"   Source      : {local_file_uri}")
        print(f"   Destination : {adls_path}")
        print(f"   Method      : dbutils.fs.cp — zero Spark processing")

    finally:
        # Restore original os.utime
        os.utime = original_utime
        print(f"✅ os.utime restored")

        # Always clean up temp dir
        shutil.rmtree(temp_dir, ignore_errors=True)
        print(f"🧹 Temp directory cleaned up: {temp_dir}")

# ── Verify file in ADLS ───────────────────────────────────────────
print(f"\n🔍 Verifying file in ADLS staging...")
partition_folder = (paths["staging"] +
    f"{dataset_folder}/year={year}/month={month}/day={day}/")
files      = dbutils.fs.ls(partition_folder)
data_files = [f for f in files if not f.name.startswith("_")
                               and not f.name.startswith(".")]

for f in data_files:
    size_mb = round(f.size / (1024*1024), 2)
    print(f"✅ {f.name} — {size_mb} MB")

# ── Show folder structure ─────────────────────────────────────────
print(f"\n📂 Staging folder structure:")
print(f"   staging/")
print(f"   └── {dataset_folder}/")
print(f"       └── year={year}/")
print(f"           └── month={month}/")
print(f"               └── day={day}/")
print(f"                   └── creditcard.csv ✅")

staged_file_adls_path = adls_path
print(f"\n📌 staged_file_adls_path → {staged_file_adls_path}")
print("\nCell 2 complete — ready for Cell 3")